# 12d - Thesis figures (from 12a / 12b / 12c CSVs)

> Copyright (C) 2024-2026 Marco Heinzen - SPDX-License-Identifier: AGPL-3.0-or-later
> Part of the Master Thesis "Building Damage Assessment with Multimodal Satellite Time Series and Machine Learning in the Russia-Ukraine War 2022-2026"
> Code hosted at https://github.com/marcoheinzen/bda
> Parts of this code were written or improved with the assistance of Claude (Anthropic); all other code, and the concept, research, architecture, design, execution, testing and validation throughout, are the author's work.


Renders the headline result figures and saves 300-dpi PNGs to `results/nb12/figures/`. Each figure is guarded: if
its input CSV is missing (upstream notebook not yet run, or run with a different SELECT), that figure is skipped.

Inputs (from `results/nb12/`): `nb12a_meanfolds_vs_pooled.csv`, `nb12a_per_city_auc_matrix.csv`,
`nb12b_meanfolds_bca_ci.csv`, `nb12c_dispatch_per_city.csv`, `nb12c_ensemble_vs_members.csv`.

Figures (main): (1) mean-folds vs pooled, (2) mean-folds + BCa CI forest, (3) per-city AUC spread, (4) experiment x city
heatmap, (5) CV-scheme leakage delta (random k-fold vs GroupKFold). Appendix (INCLUDE_APPENDIX): A1 modality dispatch, A2 city-agnostic ensemble. Figures already produced by source notebooks (label-noise 13c/M8, per-fold 13c/M3, tuning 11f, model comparison 09c/E9, DL curves 09d) are NOT remade here. Run in WSL/JupyterLab.

In [ ]:
import sys, json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt

_known = [Path("/content/drive_f/masterthesis/notebooks"),
          Path("/mnt/f/PROJECTS/masterthesis/gdrive/masterthesis/notebooks"),
          Path(r"F:\PROJECTS\masterthesis\gdrive\masterthesis\notebooks")]
_nb = next((c for c in list(Path.cwd().parents) + _known if (c / "global_setup.py").exists()), None)
if _nb is None: raise RuntimeError("global_setup.py not found")
if str(_nb) not in sys.path: sys.path.insert(0, str(_nb))
import global_setup as gs

RESULTS_ROOT = Path(gs.RESULTS_ROOT)
NB12 = RESULTS_ROOT / "nb12"
FIG_DIR = NB12 / "figures"; FIG_DIR.mkdir(parents=True, exist_ok=True)

mpl.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 300, "font.size": 10,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.3, "axes.axisbelow": True,
    "figure.facecolor": "white", "savefig.facecolor": "white", "savefig.bbox": "tight",
})
rng = np.random.default_rng(0)

def load_csv(name):
    p = NB12 / name
    if p.exists(): return pd.read_csv(p)
    print("  (missing)", name); return None

A = load_csv("nb12a_meanfolds_vs_pooled.csv")
MAT = pd.read_csv(NB12 / "nb12a_per_city_auc_matrix.csv", index_col=0) if (NB12/"nb12a_per_city_auc_matrix.csv").exists() else None
CI = load_csv("nb12b_meanfolds_bca_ci.csv")
DISP = load_csv("nb12c_dispatch_per_city.csv")
ENS = load_csv("nb12c_ensemble_vs_members.csv")

try:
    gk = json.loads(Path(getattr(gs,"GROUPKFOLD_PATH", Path(gs.STACK_ROOT)/"groupkfold_assignments.json")).read_text())
    DMG = {c: a.get("n_damage_labels", np.nan) for c,a in gk["city_assignments"].items()}
except Exception:
    DMG = {}

TOPN = 15
INCLUDE_APPENDIX = False   # True -> also render the NB12c dispatch/ensemble appendix figures (negative results)

# Curated headline set for the thesis figures (edit freely; set HEADLINE = None to fall back to top-TOPN).
# Spans the narrative: RGB leaders, SAR+optical fusion, the pooled-inflation pair (CLF_F7),
# the classical-ML best/base (NB09c), and an xBD-trained transfer model.
HEADLINE = [
    "NB08b_BDA_RGB_XGBoost",
    "NB08b_BDA_RGB_LogReg",
    "NB08b_BDA_COH_DROP+RGB+CARD_AdaBoost",
    "NB08b_BDA_COH_DROP+RGB_AdaBoost",
    "CLF_F7_LightGBM",
    "CLF_F7_GBM",
    "E7c_HistGBM_imputed",
    "E1_base_GBM",
    "NB08b_xBD_RGB_LightGBM",
]
SELECT = HEADLINE   # nb12a-based figures (1,3,4) restrict to this; None -> top-TOPN

def short_label(s, n=42):
    s = str(s); return s if len(s) <= n else s[:n-2] + ".."
def save_fig(fig, name):
    p = FIG_DIR / f"{name}.png"; fig.savefig(p); print("  saved:", p.relative_to(RESULTS_ROOT))
def pick(df, col="mean_folds_auc"):
    if SELECT: return df[df["experiment"].isin(SELECT)]
    return df.sort_values(col, ascending=False).head(TOPN)

print("inputs loaded:", {"12a":A is not None,"matrix":MAT is not None,"ci":CI is not None,
                          "dispatch":DISP is not None,"ensemble":ENS is not None})
print("figures ->", FIG_DIR)

## Fig 1 - Mean-of-folds vs pooled OOF

Shows pooled OOF sits above the honest mean-of-folds because Mariupol (41.7% of labels) dominates the pooled metric;
the hollow marker (pooled without Mariupol) moves back toward mean-folds.

In [ ]:
if A is not None:
    d = pick(A).sort_values("mean_folds_auc")
    y = np.arange(len(d))
    fig, ax = plt.subplots(figsize=(9, 0.42*len(d)+1.6))
    for yi,(mf,pl) in enumerate(zip(d["mean_folds_auc"], d["pooled_auc"])):
        ax.plot([mf,pl],[yi,yi], color="0.75", lw=2, zorder=1)
    ax.scatter(d["mean_folds_auc"], y, s=55, color="#2c7fb8", label="mean-of-folds (honest)", zorder=3)
    ax.scatter(d["pooled_auc"], y, s=55, color="#de2d26", label="pooled OOF", zorder=3)
    if "pooled_auc_no_mariupol" in d.columns:
        ax.scatter(d["pooled_auc_no_mariupol"], y, s=46, facecolors="none", edgecolors="#756bb1",
                   linewidths=1.6, label="pooled, no Mariupol", zorder=3)
    ax.axvline(0.5, color="0.5", ls="--", lw=1)
    ax.set_yticks(y); ax.set_yticklabels([short_label(e) for e in d["experiment"]])
    ax.set_xlabel("AUC"); ax.set_xlim(0.45, 1.0)
    ax.set_title("Mean-of-folds vs pooled OOF (pooled inflated by Mariupol)")
    ax.legend(loc="lower right", framealpha=0.9, fontsize=8)
    fig.tight_layout(); save_fig(fig, "fig1_meanfolds_vs_pooled"); plt.show()
else:
    print("skip fig1: run 12a")

## Fig 2 - Mean-of-folds with bootstrap BCa 95% CIs (forest plot)

The headline "which configurations are distinguishable" figure. Whiskers are the 12b BCa intervals; dashed line marks
random (0.5).

In [ ]:
if CI is not None and {"mean_folds","ci95_low","ci95_high"}.issubset(CI.columns):
    d = CI.dropna(subset=["mean_folds"]).copy()
    if SELECT:
        _h = d[d["experiment"].isin(SELECT)]
        d = _h if len(_h) >= 3 else d   # headline subset if enough are present, else all
    d = d.sort_values("mean_folds").reset_index(drop=True)
    y = np.arange(len(d))
    elo = (d["mean_folds"]-d["ci95_low"]).clip(lower=0)
    ehi = (d["ci95_high"]-d["mean_folds"]).clip(lower=0)
    fig, ax = plt.subplots(figsize=(9, 0.42*len(d)+1.6))
    ax.errorbar(d["mean_folds"], y, xerr=[elo, ehi], fmt="o", color="#2c7fb8",
                ecolor="0.5", elinewidth=1.5, capsize=3, markersize=6)
    ax.axvline(0.5, color="0.5", ls="--", lw=1, label="random (0.5)")
    ax.set_yticks(y); ax.set_yticklabels([short_label(e) for e in d["experiment"]])
    ax.set_xlabel("mean-of-folds AUC (95% BCa CI)"); ax.set_xlim(0.4, 1.0)
    ax.set_title("Cross-city performance with bootstrap confidence intervals")
    ax.legend(loc="lower right", fontsize=8)
    fig.tight_layout(); save_fig(fig, "fig2_meanfolds_bca_ci"); plt.show()
else:
    print("skip fig2: run 12b to produce nb12b_meanfolds_bca_ci.csv")

## Fig 3 - Per-city AUC spread

Distribution of per-city AUC across experiments, cities sorted by median. The wide spread (and weak cities) is the
evidence that the product is a block-level screening layer, not a per-building verdict. City labels show damage-label
counts where available.

In [ ]:
if MAT is not None:
    sub = MAT.loc[[e for e in SELECT if e in MAT.index]] if SELECT else MAT
    data = {c: sub[c].dropna().values for c in sub.columns if sub[c].notna().any()}
    order = sorted(data, key=lambda c: np.median(data[c]))
    vals = [data[c] for c in order]
    fig, ax = plt.subplots(figsize=(max(8, 0.5*len(order)+2), 5))
    bp = ax.boxplot(vals, vert=True, patch_artist=True, widths=0.6, showfliers=False)
    for b in bp["boxes"]: b.set(facecolor="#a6bddb", alpha=0.85)
    for m in bp["medians"]: m.set(color="#08519c", lw=1.5)
    for i,v in enumerate(vals, start=1):
        ax.scatter(np.full(len(v), i)+rng.uniform(-0.12,0.12,len(v)), v, s=8, color="0.3", alpha=0.5, zorder=3)
    ax.axhline(0.5, color="0.5", ls="--", lw=1)
    labs = [f"{c}\n(n={int(DMG[c])})" if (c in DMG and DMG[c]==DMG[c]) else c for c in order]
    ax.set_xticks(range(1,len(order)+1)); ax.set_xticklabels(labs, rotation=90, fontsize=8)
    ax.set_ylabel("per-city AUC"); ax.set_ylim(0.2, 1.0)
    ax.set_title("Per-city AUC across experiments (screening, not per-building)")
    fig.tight_layout(); save_fig(fig, "fig3_per_city_auc_spread"); plt.show()
else:
    print("skip fig3: run 12a")

## Fig 4 - AUC heatmap (experiment x city)

Top experiments by mean-folds against cities (ordered by mean AUC). Shows which model wins where and which cities are
hard across the board.

In [ ]:
if MAT is not None and A is not None:
    top = [e for e in pick(A)["experiment"].tolist() if e in MAT.index]
    sub = MAT.loc[top]
    city_order = sub.mean(axis=0).sort_values(ascending=False).index.tolist()
    sub = sub[city_order]
    fig, ax = plt.subplots(figsize=(max(8, 0.45*sub.shape[1]+3), 0.42*sub.shape[0]+2))
    im = ax.imshow(sub.values.astype(float), aspect="auto", cmap="RdYlGn", vmin=0.3, vmax=0.9)
    ax.set_xticks(range(sub.shape[1])); ax.set_xticklabels(city_order, rotation=90, fontsize=8)
    ax.set_yticks(range(sub.shape[0])); ax.set_yticklabels([short_label(e,38) for e in sub.index], fontsize=8)
    for i in range(sub.shape[0]):
        for j in range(sub.shape[1]):
            v = sub.values[i,j]
            if v==v: ax.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=6, color="black")
    cbar = fig.colorbar(im, ax=ax, fraction=0.025, pad=0.01); cbar.set_label("AUC")
    ax.set_title("AUC by experiment x city")
    fig.tight_layout(); save_fig(fig, "fig4_experiment_city_heatmap"); plt.show()
else:
    print("skip fig4: run 12a")

## Fig 5 - CV-scheme leakage delta (random k-fold vs GroupKFold-by-city)
The central methodological figure. Reads the latest NB11c M7 bootstrap CSVs (buildings + points) and
plots the AUC difference (random minus spatial CV) with 95% bootstrap CIs. A star marks deltas whose CI
excludes zero. This figure has no PNG produced upstream, so 12d makes it.

In [ ]:
# Fig 5 - leakage delta from NB11c M7 (glob latest buildings + points)
M7DIR = RESULTS_ROOT / "nb11c" / "cell_m7"
def _latest_csv(pat, d=M7DIR):
    fs = sorted(d.glob(pat))
    return pd.read_csv(fs[-1]) if fs else None

_lk = []
for _unit, _pat in [("buildings", "M7_leakage_delta_bootstrap_buildings_v2_*.csv"),
                    ("points",    "M7_leakage_delta_bootstrap_points_v3plus_*.csv")]:
    _df = _latest_csv(_pat)
    if _df is not None:
        _df = _df.copy(); _df["unit"] = _unit; _lk.append(_df)

if _lk:
    L = pd.concat(_lk, ignore_index=True).sort_values(["unit", "base_name"]).reset_index(drop=True)
    y = np.arange(len(L))
    elo = (L["delta_point"] - L["delta_lo"]).clip(lower=0)
    ehi = (L["delta_hi"] - L["delta_point"]).clip(lower=0)
    cols = ["#d7301f" if z else "#878787" for z in L["ci_excludes_zero"]]
    fig, ax = plt.subplots(figsize=(8, 0.55*len(L) + 1.8))
    ax.errorbar(L["delta_point"], y, xerr=[elo, ehi], fmt="none",
                ecolor="0.5", elinewidth=1.6, capsize=4, zorder=2)
    ax.scatter(L["delta_point"], y, c=cols, s=65, zorder=3, edgecolors="0.2", linewidths=0.6)
    ax.axvline(0.0, color="0.4", ls="--", lw=1)
    for yi, (dp, zz) in enumerate(zip(L["delta_point"], L["ci_excludes_zero"])):
        ax.text(dp, yi + 0.16, f"{dp:+.3f}{'*' if zz else ''}", ha="center", va="bottom", fontsize=8)
    ax.set_yticks(y); ax.set_yticklabels([f"{b}\n({u})" for b, u in zip(L["base_name"], L["unit"])], fontsize=8)
    ax.set_xlabel("AUC inflation: random k-fold minus GroupKFold-by-city (95% bootstrap CI)")
    ax.set_title("Spatial-CV leakage: random splitting inflates AUC  (* CI excludes 0)")
    fig.tight_layout(); save_fig(fig, "fig5_leakage_delta_random_vs_groupkfold"); plt.show()
else:
    print("skip fig5 (leakage): NB11c M7 CSVs not found in", M7DIR)

## Fig A1 (appendix) - Modality dispatch per city
Negative result (routing does not beat a fixed model); rendered only when INCLUDE_APPENDIX=True.

Dispatched per-city AUC (nested leave-one-city-out), each bar coloured by the model the modality rule selected for
that city. Title shows the dispatch mean-of-folds.

In [ ]:
if INCLUDE_APPENDIX and DISP is not None and {"city","chosen","auc"}.issubset(DISP.columns):
    d = DISP.sort_values("auc").reset_index(drop=True)
    models = list(dict.fromkeys(d["chosen"]))
    cmap = plt.get_cmap("tab10"); cmodel = {m: cmap(i % 10) for i,m in enumerate(models)}
    y = np.arange(len(d))
    fig, ax = plt.subplots(figsize=(9, 0.4*len(d)+1.6))
    ax.barh(y, d["auc"], color=[cmodel[m] for m in d["chosen"]])
    ax.axvline(0.5, color="0.5", ls="--", lw=1)
    if "combo" in d.columns:
        ax.set_yticklabels([f"{c} [{cb}]" for c,cb in zip(d["city"], d["combo"])], fontsize=8)
    else:
        ax.set_yticklabels(d["city"], fontsize=8)
    ax.set_yticks(y); ax.set_xlabel("dispatched per-city AUC"); ax.set_xlim(0, 1.0)
    ax.set_title(f"Modality dispatch: per-city AUC (mean = {d['auc'].mean():.3f})")
    handles = [plt.Rectangle((0,0),1,1,color=cmodel[m]) for m in models]
    ax.legend(handles, [short_label(m,30) for m in models], fontsize=7, loc="lower right", title="chosen model")
    fig.tight_layout(); save_fig(fig, "figA1_dispatch_per_city"); plt.show()
else:
    print("skip figA1 (dispatch): appendix-gated (INCLUDE_APPENDIX=False) or 12c not run")

## Fig A2 (appendix) - City-agnostic ensemble vs members
Negative result (no tightening vs members); rendered only when INCLUDE_APPENDIX=True.

Mean-of-folds per model with error bar = std across cities (the per-city spread). If the ensemble (highlighted) keeps
a similar mean with a smaller std, combination buys stability for Ukraine-wide deployment.

In [ ]:
if INCLUDE_APPENDIX and ENS is not None and {"model","mean_folds","std_folds"}.issubset(ENS.columns):
    d = ENS.sort_values("mean_folds").reset_index(drop=True)
    y = np.arange(len(d))
    colors = ["#de2d26" if m=="p_ensemble" else "#2c7fb8" for m in d["model"]]
    fig, ax = plt.subplots(figsize=(8, 0.5*len(d)+1.6))
    ax.barh(y, d["mean_folds"], xerr=d["std_folds"], color=colors, ecolor="0.4", capsize=3, alpha=0.9)
    ax.axvline(0.5, color="0.5", ls="--", lw=1)
    ax.set_yticks(y); ax.set_yticklabels([short_label(m,34) for m in d["model"]], fontsize=8)
    ax.set_xlabel("mean-of-folds AUC (error bar = std across cities)"); ax.set_xlim(0, 1.0)
    ax.set_title("City-agnostic ensemble vs members")
    fig.tight_layout(); save_fig(fig, "figA2_ensemble_vs_members"); plt.show()
else:
    print("skip figA2 (ensemble): appendix-gated (INCLUDE_APPENDIX=False) or 12c not run")

## Done

All figures written to `results/nb12/figures/` at 300 dpi. Knobs: `TOPN` (how many experiments in the nb12a-based
figures) and `SELECT` (restrict to an explicit experiment list, matching what you set in 12b/12c). Re-run after the
9d/10d/11x reruns + 12a/12b/12c so the figures reflect final numbers.

## Figures sourced from other notebooks (do not remake here)
These thesis figures are already rendered by their source notebooks; pick the latest-timestamp PNG:
- Per-fold AUC boxplot, fold x model heatmap -> `results/nb11c/cell_m3/`
- Label-noise stability (buildings vs points) -> `results/nb11c/cell_m8/`
- Tuning lift boxplots / param distributions -> `results/nb11f/cell_m2/`, `cell_m6/`
- Model comparison bar (classical ML) -> `results/nb09c_v2/cell_e9/`
- DL training curves / spatial CM -> `results/nb09d_dl_experiments/plots/`

Rerun order next week: 9d/10d done -> 12a/12b/12c -> 13c (M7/M8) -> then this notebook, so figures reflect final numbers.